In [1]:
import numpy as np
from ase.io import read, write

# 1. Đọc file .cif Spinel nền (mp-18748 / Co3O4)
atoms = read('Co3O4.cif')

# 2. Cập nhật hằng số mạng a = b = c = 8.15 Å từ kết quả XRD/Rietveld
atoms.set_cell([8.15, 8.15, 8.15, 90, 90, 90], scale_atoms=True)

# 3. Lọc danh sách chỉ số (indices) của 24 vị trí kim loại (Co)
metal_indices = [atom.index for atom in atoms if atom.symbol == 'Co']

# 4. Khởi tạo danh sách 24 kim loại theo tỷ lệ (5 Fe, 5 Mn, 5 Co, 5 Ni, 4 Cu)
metals = ['Fe']*5 + ['Mn']*5 + ['Co']*5 + ['Ni']*5 + ['Cu']*4

# Trộn ngẫu nhiên phân bố các nguyên tử kim loại
np.random.seed(42)  # Cố định seed để có thể tái lập chính xác mô hình này sau này
np.random.shuffle(metals)

# Gán lại ký hiệu hóa học cho 24 vị trí kim loại
for idx, new_symbol in zip(metal_indices, metals):
    atoms[idx].symbol = new_symbol

# 5. Khai báo danh sách các file thế giả UPF trong thư mục ../pseudos
pseudos = {
    'Fe': 'Fe.paw.pbe.z_16.ld1.psl.v0.2.1.upf',
    'Mn': 'Mn.us.pbe.z_15.uspp.gbrv.v1.5.upf',
    'Co': 'Co.nc.pbe.z_17.oncvpsp4.spms.v1.upf',
    'Ni': 'Ni.nc.pbe.z_18.oncvpsp4.spms.v1.upf',
    'Cu': 'Cu.paw.pbe.z_11.ld1.psl.v1.0.0-low.upf',
    'O':  'O.paw.pbe.z_6.ld1.psl.v0.1.upf'
}

# 6. Cấu hình thông số Namelist cho Quantum ESPRESSO
input_data = {
    'control': {
        'calculation': 'vc-relax',
        'restart_mode': 'from_scratch',
        'prefix': 'HEMMO_Spinel',
        'pseudo_dir': '../pseudos',
        'outdir': './tmp'
    },
    'system': {
        'ibrav': 0,
        'ecutwfc': 50.0,
        'ecutrho': 400.0,
        'occupations': 'smearing',
        'smearing': 'mv',
        'degauss': 0.01,
        'nspin': 2
    },
    'electrons': {
        'conv_thr': 1.0e-6,
        'mixing_beta': 0.3
    },
    'ions': {
        'ion_dynamics': 'bfgs'
    },
    'cell': {
        'cell_dynamics': 'bfgs'
    }
}

# 7. Xuất trực tiếp ra file 1_pw.in
write('1_pw.in', atoms, format='espresso-in',
      input_data=input_data,
      pseudopotentials=pseudos,
      kpts=(3, 3, 3))

# Xuất thêm 1 file .cif mới để mở nhanh trên VESTA kiểm tra lại giao diện 3D
write('HEMMO_Spinel_56atoms.cif', atoms)

print("-> Đã tạo thành công '1_pw.in' và 'HEMMO_Spinel_56atoms.cif'!")

-> Đã tạo thành công '1_pw.in' và 'HEMMO_Spinel_56atoms.cif'!


/home/tripc/DE_AN/.venv/lib/python3.14/site-packages/ase/io/cif.py:837: UserWarning: Occupancies present but no occupancy info for "{symbol}"
  warnings.warn(str(err))
